# Réseau de neurones simple avec PyTorch

Dans ce TP, nous allons construire plusieurs réseaux de neurones : un simple d'abord (sans couche cachée), puis un perceptron multicouches (plusieurs couches cachées). Nous utiliserons le dataset MNIST pour tester l'apprentissage de ce réseau en conditions réelles.

MNIST est le dataset *Hello World!* du machine learning. Il est composé d'images de 28 $\times$ 28 pixels en niveaux de gris. Ces images représentent des chiffres (0 à 9). Chaque image est associée à un label indiquant le caractère que l'image est sensée représenter.

## Imports des librairies

PyTorch et le dataset MNIST


In [ ]:
import collections.abc
import random

import numpy
import matplotlib.pyplot as plt
import seaborn
import torch
import torchvision
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

# Les calculs auront lieu sur le GPU s'il y en a un
device = "cuda" if torch.cuda.is_available() else "cpu"

[`matplotlib`](https://matplotlib.org/), [`numpy`](https://numpy.org/) et [`seaborn`](https://seaborn.pydata.org/) sont des librairies de base en machine learning avec Python que nous utiliserons ici pour la visualisation et les opérations simples sur des matrices

## Récupération des données

In [ ]:
train_data = torchvision.datasets.MNIST("data", train=True, download=True)
test_data = torchvision.datasets.MNIST("data", train=False, download=True)
X_train, y_train = train_data.data, train_data.targets
X_test, y_test = test_data.data, test_data.targets

## Regardons les données

Utilisez la cellule suivante pour explorer les tenseurs `X_train`, `y_train`, `X_test` et `y_test`. N'hésitez pas à utiliser la complétion automatique pour parcourir leurs attributs et méthodes.

Afin de faire du machine learning, on cherche des ensembles de train et de test et à afficher quelques exemples avec leurs labels associés.

Vous essayerez de répondre aux questions suivantes :
- Combien y a-t-il d'images au total dans ce dataset ?
- Sous quelle forme sont stockées les images ? les labels ?
- Les classes sont-elles équilibrées ?

In [ ]:
# Votre code ici

### Solution

In [ ]:
example = X_train[0]
example_label = int(y_train[0])

print(f"Format des exemples : {tuple(X_train.shape)}")
print(f"Format des labels : {tuple(y_train.shape)}")

plt.imshow(example, cmap="gray_r")
plt.title(f"Premier exemple du dataset ({example_label})")
plt.show()

seaborn.countplot(x=y_train.numpy())
plt.title("Décompte des différentes classes (chiffres)")
plt.ylabel("Décompte")
plt.xlabel("Chiffre")
plt.show()

## Affichage d'exemples

Affichez 25 exemples, tirés au hasard dans la base de train. Vous n'oublierez pas d'afficher le label correspondant sous une forme facile à lire.

Pour cela utilisez :
- [`numpy.random.choice`](https://numpy.org/doc/stable/reference/random/generated/numpy.random.choice.html)
- [`matplotlib.pyplot.imshow`](https://matplotlib.org/api/_as_gen/matplotlib.pyplot.imshow.html)
- [`matplotlib.pyplot.subplots`](https://matplotlib.org/api/_as_gen/matplotlib.pyplot.subplots.html) pour un affichage élégant

In [ ]:
# Utilisation des subplots pour un affichage en grande grille 5x5
f, ax = plt.subplots(5, 5, figsize=(15, 15))

# Votre code ici

# ax[2, 4].imshow(... , cmap="gray_r")
# ax[2, 4].set_title("Exemple n (label)")

### Solution

In [ ]:
# Création de la grille de sous-plots. On donne l'argument figsize pour agrandir
# la taille de la figure qui est petite par défaut
f, ax = plt.subplots(5, 5, figsize=(15, 15))

# On choisit 25 indices au hasard, sans replacement (on ne veut pas afficher la
# même image deux fois)
random_indexes = numpy.random.choice(X_train.shape[0],
                                     size=(5, 5),
                                     replace=False)

for i in range(5):
  for j in range(5):
    img_index = random_indexes[i, j]
    image = X_train[img_index]
    label = int(y_train[img_index])

    # Affichage avec matplotlib et sa fonction imshow, très pratique en vision par
    # ordinateur
    ax[i, j].imshow(image, cmap='gray_r')
    ax[i, j].set_title(f"Exemple {img_index} ({label})")
    ax[i, j].axis('off')

## Transformation des données

Nous devons effectuer quelques transformations sur ces données :

1. On pourrait conserver les formes originales des tenseurs `(_, 28, 28)` mais il sera plus aisé de travailler sur des tenseurs de forme `(_, 28²)` où `_` est le nombre original d'exemples.
2. On souhaite utiliser l'intervalle $[0, 1]$ plutôt que des valeurs entre $0$ et $255$ pour nos images. Transformez les entrées pour cela.

Fonction utile : [`torch.Tensor.reshape`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.reshape.html)

*Effectuez les deux premières transformations sur `X_train` et `X_test`.*

In [ ]:
# Votre code ici

### Solution

In [ ]:
nb_classes = 10
input_dim = 28 * 28

# On veut mettre X « à plat » pour que l'input de notre réseau soit un vecteur
# de taille input_dim
X_train = X_train.reshape(X_train.shape[0], input_dim)
#syntaxe  équivalente
X_train = X_train.reshape(-1, input_dim)

X_test = X_test.reshape(X_test.shape[0], input_dim)
#syntaxe équivalente
X_test = X_test.reshape(-1, input_dim)

# Transformation vers [0, 1]
X_train = X_train / 255.0
X_test = X_test / 255.0

# Déplacement des tenseurs sur le périphérique de calcul
X_train, y_train = X_train.to(device), y_train.to(device)
X_test, y_test = X_test.to(device), y_test.to(device)

## Séparation entraînement / validation

Pour choisir les hyper-paramètres (learning rate, taille du réseau…), on compare des modèles sur un ensemble de **validation**. L'ensemble de test est réservé à l'évaluation finale : s'en servir pour faire des choix rendrait cette évaluation trop optimiste.

On met de côté les 10 000 derniers exemples d'entraînement pour la validation :

In [ ]:
X_train, X_val = X_train[:50000], X_train[50000:]
y_train, y_val = y_train[:50000], y_train[50000:]
print(X_train.shape, X_val.shape, X_test.shape)

## Construction d'un réseau de neurones sans couche cachée

### Variables du modèle

Dans un réseau de neurones sans couche cachée, $y = \sigma(XW +b)$, où $\sigma$ est une fonction adaptée au problème. Ici, on choisira la fonction softmax, étant donné que nous nous attaquons à un problème de classification multi-classes.

Note sur les variables utilisées : dans le cours, on utilise $\theta$ pour dénoter tous les poids : ceux qu'on multiplie à l'entrée ansi que ceux qu'on ajoute — les biais). Dans ce TPs, on dénote donc les premiers `W` et les seconds `b`, c'est une notation très courante.

*Complétez la fonction `create_simple_nn_weights` pour initialiser `W` et `b` comme [paramètres PyTorch](https://docs.pytorch.org/docs/stable/generated/torch.nn.parameter.Parameter.html) avec la fonction [`torch.randn`](https://docs.pytorch.org/docs/stable/generated/torch.randn.html).*

In [ ]:
def create_simple_nn_weights(n_inputs: int,
                             n_outputs: int
                             ) -> tuple[nn.Parameter, nn.Parameter]:
  W = None  # Votre code ici
  b = None  # Votre code ici
  return W, b

#### Solution

In [ ]:
def create_simple_nn_weights(n_inputs: int,
                             n_outputs: int
                             ) -> tuple[nn.Parameter, nn.Parameter]:
  return (nn.Parameter(torch.randn(n_inputs, n_outputs, device=device)),
          nn.Parameter(torch.randn(n_outputs, device=device)))

### Définition du modèle

Nous allons ici préparer l'architecture de notre modèle.

*Corrigez le code suivant afin que `y_pred` soit le résultat de notre réseau de neurones sans couches cachées : $\text{softmax}(XW+b)$. On utilisera le [log du softmax](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.log_softmax.html) plutôt que le softmax standard pour plus de stabilité numérique.*

In [ ]:
class SimpleNN(nn.Module):
  def __init__(self, n_inputs: int = 28 ** 2, n_outputs: int = 10) -> None:
    super().__init__()
    self.W, self.b = create_simple_nn_weights(n_inputs, n_outputs)

  def forward(self, X: torch.Tensor) -> torch.Tensor:
    y_pred = X  # Votre code ici
    return y_pred


simple_nn = SimpleNN()
simple_nn(X_train)

#### Solution

In [ ]:
# Creation du modèle
class SimpleNN(nn.Module):
  def __init__(self, n_inputs: int = 28 ** 2, n_outputs: int = 10) -> None:
    super().__init__()
    self.W, self.b = create_simple_nn_weights(n_inputs, n_outputs)

  def forward(self, X: torch.Tensor) -> torch.Tensor:
    return torch.log_softmax(X @ self.W + self.b, dim=-1)


simple_nn = SimpleNN()
simple_nn(X_train)

### Calcul de la fonction de perte

Nous allons utiliser une fonction de perte standard en classification multi-classes : l'entropie croisée. Notre modèle renvoyant déjà des log-probabilités, vous pouvez faire appel à [`torch.nn.functional.nll_loss`](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.nll_loss.html). De plus on souhaite pouvoir interpréter directement cette valeur, il faut donc que la fonction `categorical_crossentropy` rende une valeur aggrégée (la moyenne) : appelez `nll_loss` avec `reduction="none"`, qui rend une valeur par exemple.

Vous pourrez utiliser pour cela [`torch.mean`](https://docs.pytorch.org/docs/stable/generated/torch.mean.html).

*Complétez la fonction `categorical_crossentropy`.*

In [ ]:
def categorical_crossentropy(y: torch.Tensor,
                             y_pred: torch.Tensor) -> torch.Tensor:
  loss = None  # Votre code ici
  return loss


simple_nn = SimpleNN()
y_train_pred = simple_nn(X_train)
categorical_crossentropy(y_train, y_train_pred)

#### Solution

In [ ]:
def categorical_crossentropy(y: torch.Tensor,
                             y_pred: torch.Tensor) -> torch.Tensor:
  cross_entropy = nn.functional.nll_loss(y_pred, y, reduction="none")
  return torch.mean(cross_entropy)


simple_nn = SimpleNN()
y_train_pred = simple_nn(X_train)
categorical_crossentropy(y_train, y_train_pred)

### Métriques

Pour savoir si un modèle apprend correctement, il est important de mesurer ses performances. Dans ces travaux pratiques, nous allons utiliser la performance la plus simple mais aussi une des plus informatives : l'accuracy. C'est simplement le nombre de bonnes prédictions sur le nombre total de prédictions.

*Utilisez [torch.argmax](https://docs.pytorch.org/docs/stable/generated/torch.argmax.html) et les comparaisons de tenseurs pour compléter la fonction accuracy.*

In [ ]:
def accuracy(y: torch.Tensor, y_pred: torch.Tensor) -> float:
  predictions = None  # Votre code ici
  return None


simple_nn = SimpleNN()
y_test_pred = simple_nn(X_test)
accuracy(y_test, y_test_pred)

#### Solution

In [ ]:
def accuracy(y: torch.Tensor, y_pred: torch.Tensor) -> float:
  argmaxed = torch.argmax(y_pred, dim=-1)
  n_equal = torch.count_nonzero(argmaxed == y)
  return (n_equal / y.shape[0]).item()


simple_nn = SimpleNN()
y_test_pred = simple_nn(X_test)
accuracy(y_test, y_test_pred)

## Création d'un `DataLoader` PyTorch

- *Utilisez [`torch.utils.data.TensorDataset`](https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.TensorDataset) pour coder la fonction `create_dataset` qui crée un dataset PyTorch à partir des tenseurs `X_train` et `y_train`. Nous souhaitons obtenir des batchs sous la forme `(X_batch, y_batch)`.*
- *Enveloppez ce dataset dans un [`DataLoader`](https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader) pour regrouper les exemples en batchs.*
- *Utilisez l'argument `shuffle` pour que le dataset soit mélangé à chaque itération.*

In [ ]:
def create_dataset(X: torch.Tensor, y: torch.Tensor, batch_size: int = 4000
                   ) -> DataLoader:
  pass  # Votre code ici

### Solution

In [ ]:
def create_dataset(X: torch.Tensor, y: torch.Tensor, batch_size: int = 4000
                   ) -> DataLoader:
  dataset = TensorDataset(X, y)
  dataset = DataLoader(dataset, batch_size=batch_size, shuffle=True)
  return dataset

## Apprentissage

Voici une boucle d'apprentissage quasiment mise en place.

*Implémentez la partie manquante, qui procède à la mise à jour des paramètres pour un batch donné. Vous utiliserez pour cela l'[autodifférenciation de PyTorch](https://docs.pytorch.org/docs/stable/notes/autograd.html) : `loss.backward()` calcule le gradient de la perte et le range dans l'attribut `.grad` de chaque paramètre.*

In [ ]:
LossFunction = collections.abc.Callable[[torch.Tensor, torch.Tensor],
                                       torch.Tensor]
RegularizationFunction = collections.abc.Callable[[nn.Module], torch.Tensor]


def train(model: nn.Module,
          X_train: torch.Tensor = X_train,
          y_train: torch.Tensor = y_train,
          X_val: torch.Tensor = X_val,
          y_val: torch.Tensor = y_val,
          epochs: int = 150,
          batch_size: int = 4000,
          learning_rate: float = 0.001,
          evaluate_every: int = 10,
          loss: LossFunction = categorical_crossentropy,
          regularization: RegularizationFunction | None = None,
         ) -> tuple[list[float], ...]:
  with torch.no_grad():
    y_train_pred = model(X_train)
    y_val_pred = model(X_val)
    accuracies = [accuracy(y_train, y_train_pred)]
    val_accuracies = [accuracy(y_val, y_val_pred)]
    losses = [float(loss(y_train, y_train_pred))]
    val_losses = [float(loss(y_val, y_val_pred))]

  print(f"Métriques initiales : acc {accuracies[-1]:.4f}, "
        f"val_acc {val_accuracies[-1]:.4f}, "
        f"loss {losses[-1]:.4f}, "
        f"val_loss {val_losses[-1]:.4f}")

  dataset_train = create_dataset(X_train, y_train, batch_size)

  for e in range(epochs):

    for X_train_batch, y_train_batch in dataset_train:

      pass # Votre code ici

    # Calcul et affichage de la métrique d'évaluation sur l'ensemble de
    # validation toutes les `evaluate_every` epochs
    if (e + 1) % evaluate_every == 0:
      with torch.no_grad():
        y_train_pred = model(X_train)
        y_val_pred = model(X_val)
        accuracies.append(accuracy(y_train, y_train_pred))
        losses.append(float(loss(y_train, y_train_pred)))
        val_accuracies.append(accuracy(y_val, y_val_pred))
        val_losses.append(float(loss(y_val, y_val_pred)))
      print(f"Métriques epoch {e + 1} : acc {accuracies[-1]:.4f}, "
            f"val_acc {val_accuracies[-1]:.4f}, "
            f"loss {losses[-1]:.4f}, "
            f"val_loss {val_losses[-1]:.4f}")

  x_ticks = numpy.arange(0, epochs + 1, evaluate_every)

  plt.plot(x_ticks, losses, label="Entraînement")
  plt.plot(x_ticks, val_losses, label="Validation")
  plt.title("Fonction de perte pendant l'entraînement")
  plt.xlabel("Epochs")
  plt.legend(loc="upper right")
  plt.show()
  plt.plot(x_ticks, accuracies, label="Entraînement")
  plt.plot(x_ticks, val_accuracies, label="Validation")
  plt.title("Justesse pendant l'entraînement")
  plt.xlabel("Epochs")
  plt.legend(loc="upper right")
  plt.show()
  return accuracies, val_accuracies, losses, val_losses


simple_nn = SimpleNN()
_ = train(simple_nn, epochs=100, learning_rate=3)

### Solution

In [ ]:
LossFunction = collections.abc.Callable[[torch.Tensor, torch.Tensor],
                                       torch.Tensor]
RegularizationFunction = collections.abc.Callable[[nn.Module], torch.Tensor]


def train(model: nn.Module,
          X_train: torch.Tensor = X_train,
          y_train: torch.Tensor = y_train,
          X_val: torch.Tensor = X_val,
          y_val: torch.Tensor = y_val,
          epochs: int = 150,
          batch_size: int = 4000,
          learning_rate: float = 0.001,
          evaluate_every: int = 10,
          loss: LossFunction = categorical_crossentropy,
          regularization: RegularizationFunction | None = None,
         ) -> tuple[list[float], ...]:
  with torch.no_grad():
    y_train_pred = model(X_train)
    y_val_pred = model(X_val)
    accuracies = [accuracy(y_train, y_train_pred)]
    val_accuracies = [accuracy(y_val, y_val_pred)]
    losses = [float(loss(y_train, y_train_pred))]
    val_losses = [float(loss(y_val, y_val_pred))]

  print(f"Métriques initiales : acc {accuracies[-1]:.4f}, "
        f"val_acc {val_accuracies[-1]:.4f}, "
        f"loss {losses[-1]:.4f}, "
        f"val_loss {val_losses[-1]:.4f}")

  dataset_train = create_dataset(X_train, y_train, batch_size)

  for e in range(epochs):

    for X_train_batch, y_train_batch in dataset_train:

      # Calcul de la perte
      y_train_batch_pred = model(X_train_batch)
      loss_scalar = loss(y_train_batch, y_train_batch_pred)
      if regularization is not None:
        loss_scalar = loss_scalar + regularization(model)

      # Calcul automatique du gradient de la perte par rapport à chaque
      # paramètre : il est rangé dans l'attribut .grad du paramètre
      model.zero_grad()
      loss_scalar.backward()

      # Parcours des paramètres un par un pour les mettre à jour en suivant la
      # règle : nouvelle_valeur = ancienne_valeur - learning_rate * gradient
      with torch.no_grad():
        for parameter in model.parameters():
          parameter -= parameter.grad * learning_rate

    # Calcul et affichage de la métrique d'évaluation sur l'ensemble de
    # validation toutes les `evaluate_every` epochs
    if (e + 1) % evaluate_every == 0:
      with torch.no_grad():
        y_train_pred = model(X_train)
        y_val_pred = model(X_val)
        accuracies.append(accuracy(y_train, y_train_pred))
        losses.append(float(loss(y_train, y_train_pred)))
        val_accuracies.append(accuracy(y_val, y_val_pred))
        val_losses.append(float(loss(y_val, y_val_pred)))
      print(f"Métriques epoch {e + 1} : acc {accuracies[-1]:.4f}, "
            f"val_acc {val_accuracies[-1]:.4f}, "
            f"loss {losses[-1]:.4f}, "
            f"val_loss {val_losses[-1]:.4f}")

  x_ticks = numpy.arange(0, epochs + 1, evaluate_every)

  plt.plot(x_ticks, losses, label="Entraînement")
  plt.plot(x_ticks, val_losses, label="Validation")
  plt.title("Fonction de perte pendant l'entraînement")
  plt.xlabel("Epochs")
  plt.legend(loc="upper right")
  plt.show()
  plt.plot(x_ticks, accuracies, label="Entraînement")
  plt.plot(x_ticks, val_accuracies, label="Validation")
  plt.title("Justesse pendant l'entraînement")
  plt.xlabel("Epochs")
  plt.legend(loc="upper right")
  plt.show()
  return accuracies, val_accuracies, losses, val_losses


simple_nn = SimpleNN()
_ = train(simple_nn, epochs=100, learning_rate=3)

## Passage à des réseaux profonds

Pour passer à des réseaux profonds, on ajoute des matrices de poids et de biais pour chaque couche à notre fonction de création de poids, par exemple comme suit :

In [ ]:
def create_mlp_weights(n_inputs: int,
                       n_outputs: int,
                       hidden_layer_sizes: list[int] = [512],
                      ) -> nn.ModuleList:
  variables = nn.ModuleList()

  # On garde en mémoire la dernière taille de sortie : ça sera la nouvelle
  # taille d'entrée. Vaut n_inputs au départ
  last_size = n_inputs

  for i, hidden_layer_size in enumerate(hidden_layer_sizes, 1):
    # On initialise aléatoirement une matrice de poids et un vecteur de biais.
    # La variance des poids est de 1 / (nombre d'entrées) : avec une variance de
    # 1, les activations tanh satureraient (voir le cours sur l'initialisation)
    W = torch.randn(last_size, hidden_layer_size, device=device) / last_size ** 0.5
    b = torch.zeros(hidden_layer_size, device=device)

    # On ajoute ces deux paramètres dans un dictionnaire puis dans la liste de
    # nos paramètres
    variables.append(nn.ParameterDict(dict(W=nn.Parameter(W),
                                           b=nn.Parameter(b))))

    # On met à jour la dernière taille de sortie utilisée
    last_size = hidden_layer_size

  # La dernière couche est spéciale : elle fait toujours n_outputs en taille de
  # sortie, on la traite donc à part
  W = torch.randn(last_size, n_outputs, device=device) / last_size ** 0.5
  b = torch.zeros(n_outputs, device=device)
  variables.append(nn.ParameterDict(dict(W=nn.Parameter(W),
                                         b=nn.Parameter(b))))
  return variables

Pour ce qui est de la sortie du modèle, elle est maintenant calculée itérativement, couche par couche. Chaque couche prend en entrée le résultat de la couche précédente. Par simplicité, vous pourrez fixer vous même la fonction d'activation que vous utiliserez pour les couches cachées.

*Complétez la fonction `MLP.forward` pour calculer le résultat d'un réseau profond. Chaque paramètre est accessible grâce à `self.weights[layer_number][variable_name]`. Par exemple, pour accéder à la matrice `W` du premier layer : `self.weights[0]["W"]`.*

In [ ]:
class MLP(nn.Module):
  def __init__(self,
               hidden_layer_sizes: list[int] = [512],
               n_inputs: int = 28 ** 2,
               n_outputs: int = 10) -> None:
    super().__init__()
    self.weights = create_mlp_weights(n_inputs, n_outputs, hidden_layer_sizes)

  def forward(self, X: torch.Tensor) -> torch.Tensor:
    # Votre code ici
    return X

### Solution

In [ ]:
class MLP(nn.Module):
  def __init__(self,
               hidden_layer_sizes: list[int] = [512],
               n_inputs: int = 28 ** 2,
               n_outputs: int = 10) -> None:
    super().__init__()
    self.weights = create_mlp_weights(n_inputs, n_outputs, hidden_layer_sizes)

  def forward(self, X: torch.Tensor) -> torch.Tensor:
    current = X
    for layer_weights in self.weights:
      # Calcul de la sortie de la couche de neurones non activée
      current = current @ layer_weights["W"] + layer_weights["b"]
      # Activation si on n'est pas dans la couche de sortie
      if layer_weights is not self.weights[-1]:
        current = torch.tanh(current)
    return torch.log_softmax(current, dim=-1)

## Entraînement du réseau profond

In [ ]:
_ = train(MLP(), learning_rate=0.5, epochs=100)

*Que constatez-vous en entraînant un réseau avec une couche cachée de 512 neurones ?*

Votre réponse ici

### Solution

Le réseau à une couche cachée fait nettement mieux que le réseau sans couche cachée : environ 97 % de justesse en validation, contre 88 %. Les justesses d'entraînement et de validation restent proches : pas de sur-apprentissage marqué.

C'est en partie grâce à l'initialisation des poids, de variance 1 / (nombre d'entrées). Pour vous en convaincre, retirez la division par `last_size ** 0.5` dans `create_mlp_weights` et relancez : avec des poids de variance 1, les activations tanh saturent, et le réseau atteint environ 98 % sur l'entraînement mais seulement 90 % en validation. Ce grand écart ressemble à du sur-apprentissage, mais il vient d'une mauvaise initialisation.

## Régularisation

La régularisation limite le sur-apprentissage. Elle devient utile quand le réseau est grand par rapport aux données, ou quand on l'entraîne longtemps : ici, entraînez sur 200 epochs et comparez avec l'entraînement précédent.

Pour la mettre en place, on rajoute un terme à la fonction de perte qui pénalise les valeurs importantes des poids.

*Modifiez la fonction ci-dessous pour calculer la pénalité L2. Elle est égale à la somme des carrés des paramètres.*

In [ ]:
def l2(lambda_coefficient: float) -> RegularizationFunction:
  def worker(model: nn.Module) -> torch.Tensor:
    weights_norm = torch.zeros((), device=device)
    # Votre code ici
    return lambda_coefficient * weights_norm

  return worker


train(MLP(),
      epochs=200,
      learning_rate=3e-1,
      regularization=l2(1e-4))

### Solution

In [ ]:
def l2(lambda_coefficient: float) -> RegularizationFunction:
  def worker(model: nn.Module) -> torch.Tensor:
    weights_norm = torch.zeros((), device=device)
    for parameter in model.parameters():
      weights_norm = weights_norm + torch.sum(torch.square(parameter))
    return lambda_coefficient * weights_norm

  return worker


train(MLP(),
      epochs=200,
      learning_rate=3e-1,
      regularization=l2(1e-4))  # avec 3e-3, la pénalité est trop forte : sous-apprentissage

## Recherche d'hyper-paramètres

Nous allons maintenant procéder à la recherche d'hyper-paramètres. Utilisez [`random.choice`](https://docs.python.org/fr/3/library/random.html#random.choice) pour échantillonner le learning rate et le nombre d'unités cachées et renvoyez une liste des paramètres et d'une métrique au choix.

In [ ]:
def search_hyperparameters(
    X_train: torch.Tensor = X_train,
    y_train: torch.Tensor = y_train,
    X_val: torch.Tensor = X_val,
    y_val: torch.Tensor = y_val,
    learning_rates: list[float] = [0.1, 0.5, 1],
    ns_hidden_units: list[int]  = range(32, 513, 32),
    n_iters: int = 5
    ) -> tuple[list[dict[str, float]], list[float]]:
  params = []
  val_accs = []
  for _ in range(n_iters):
    pass
  return params, val_accs


search_hyperparameters()

### Solution

In [ ]:
def search_hyperparameters(
    X_train: torch.Tensor = X_train,
    y_train: torch.Tensor = y_train,
    X_val: torch.Tensor = X_val,
    y_val: torch.Tensor = y_val,
    learning_rates: list[float] = [0.1, 0.5, 1],
    ns_hidden_units: list[int]  = range(32, 513, 32),
    n_iters: int = 5
    ) -> tuple[list[dict[str, float]], list[float]]:
  params = []
  val_accs = []
  for _ in range(n_iters):
    learning_rate = random.choice(learning_rates)
    n_hidden_units = random.choice(ns_hidden_units)
    mlp = MLP(hidden_layer_sizes=[n_hidden_units])
    _, val_acc_epochs, _, _ = train(mlp, X_train, y_train, X_val, y_val,
                                    learning_rate=learning_rate, epochs=50)
    val_accs.append(val_acc_epochs[-1])
    params.append(dict(learning_rate=learning_rate,
                       n_hidden_units=n_hidden_units))
  return params, val_accs


search_hyperparameters()

## Évaluation finale sur l'ensemble de test

L'ensemble de test n'a servi à rien jusqu'ici : ni à l'entraînement, ni au choix des hyper-paramètres. C'est ce qui rend son évaluation fiable.

*Entraînez un MLP avec les meilleurs hyper-paramètres trouvés, puis calculez sa justesse sur `X_test`. Est-elle proche de la justesse de validation ?*

In [ ]:
# Votre code ici

### Solution

In [ ]:
# Meilleure configuration trouvée (à adapter selon vos résultats)
mlp = MLP(hidden_layer_sizes=[512])
_ = train(mlp, learning_rate=0.5, epochs=100)
with torch.no_grad():
  test_accuracy = accuracy(y_test, mlp(X_test))
print(f"Justesse sur l'ensemble de test : {test_accuracy:.4f}")

La justesse de test (environ 96,5 %) est proche de la justesse de validation (environ 96,8 %). Comme l'ensemble de test n'a servi à aucun choix, c'est une estimation fiable des performances du modèle sur de nouvelles données.